# HoMM3 GPU Pose Solver V2

A new lane beside `roc_gpu_cascade_experiment.ipynb`. V2 uses exact CUDA triangle rasterization, separates static registration from per-frame motion, keeps competing hypotheses, and scores global/dense semantics only on finalists. Authentic DEF frames are required; RIFE directories are rejected and alpha-shadow culling is mandatory.

In [ ]:
from pathlib import Path
import json, subprocess, sys
from IPython.display import display, Image as DisplayImage
ROOT = Path(r'V:/Games/Olden Era Playtest Tweak')
TOOL = ROOT / 'tools' / 'homm3_silhouette_pose_fit'
THIRD_PARTY = TOOL / 'third_party' / 'gpu_pose_v2'
for _path in (THIRD_PARTY, TOOL):
    if str(_path) not in sys.path: sys.path.insert(0, str(_path))
from gpu_pose_v2 import compare_preview_frames, ensure_huggingface_auth, export, preflight, preset_config, prepare_experiment, preview, registration_preview, solve_clip
from gpu_pose_v2.blender_export import export_blender_60fps, require_blender_parity
V2_HF_TOKEN_PRESENT = ensure_huggingface_auth()
V2_PREFLIGHT = preflight(require_renderer=True, model_ids=('google/siglip2-base-patch16-512', 'facebook/dinov3-vits16-pretrain-lvd1689m'))
display({'huggingfaceTokenPresent': V2_HF_TOKEN_PRESENT, 'preflight': V2_PREFLIGHT})


In [ ]:
# Existing picker: all discovered units, animations, custom meshes, and original DEF frames.
import importlib
import materialize_rife_source_frames as _materialize_rife_source_frames
importlib.reload(_materialize_rife_source_frames)
import roc_semantic_hybrid_widgets as _asset_widgets
importlib.reload(_asset_widgets)
from roc_semantic_hybrid_widgets import gpu_pose_v2_selection, make_asset_picker
ASSET_PICKER = make_asset_picker(ROOT)
display(ASSET_PICKER.widget)
def _current_v2_selection():
    # Read the live widget values; do not freeze the initial Roc choice.
    return gpu_pose_v2_selection(ASSET_PICKER)
V2_SELECTION = _current_v2_selection()
print('selected:', ASSET_PICKER.selection_summary())


## Meshy rigidization pass

This is intentionally a separate cell. It does not silently run inside preparation. The smart rig remains the source mesh; the report is consumed as a constraint source.

In [ ]:
V2_SELECTION = _current_v2_selection()
MESH = Path(V2_SELECTION['mesh_path']).resolve()
RIGID_DIR = Path(V2_SELECTION['artifact_dir']) / 'rigidization'
RIGID_DIR.mkdir(parents=True, exist_ok=True)
RIGIDIZED_MESH = RIGID_DIR / f'{MESH.stem}_rigidized.glb'
RIGIDIZATION_REPORT = RIGID_DIR / 'meshy_rigidization_report.json'
if not RIGIDIZATION_REPORT.is_file() or not RIGIDIZED_MESH.is_file():
    subprocess.run([sys.executable, str(TOOL / 'run_meshy_rigidization.py'), '--src', str(MESH), '--out', str(RIGIDIZED_MESH), '--report', str(RIGIDIZATION_REPORT), '--overwrite'], cwd=str(ROOT), check=True)
V2_SELECTION['rigidization_report'] = RIGIDIZATION_REPORT
_V2_RIGIDIZATION_SOURCE_MESH = MESH
if RIGIDIZED_MESH.is_file():
    # The derived GLB preserves Meshy's skeleton but carries the report's
    # accepted rigid skin weights into the exact V2 geometry path.
    V2_SELECTION['mesh_path'] = RIGIDIZED_MESH
print({'sourceMesh': str(MESH), 'preparedMesh': str(V2_SELECTION['mesh_path']), 'rigidizationReport': str(RIGIDIZATION_REPORT), 'reportExists': RIGIDIZATION_REPORT.is_file()})


In [ ]:
# Tunable surface. Geometry, pose-basis, and worker details stay in imported source.
V2_PRESET = 'fast'  # short pilot; switch to 'balanced' or 'plausible_3d' after the mechanics pass
V2_DENSE_ENCODER = 'dinov3'  # dinov3 uses the already-authorized local HF cache; siglip2 is the dense ablation
V2_OVERRIDES = {
    'mesh_scale': 1.0,  # static mesh scale before registration
    'registration_frame_position': 0,  # idle/bind-like prepared frame used for static registration
    'registration_hypotheses': 96,  # one-shot coarse registrations; only the keepers are optimized
    'registration_keep': 8,  # registrations carried into articulation
    'articulated_keep': 6,  # geometric finalists before semantics
    'semantic_keep': 4,  # semantic finalist retained for the pilot
    'registration_iterations': 32,  # Adam steps for static registration
    'motion_root_iterations': 32,  # root-only steps after registration; no joints are active
    'motion_root_hypotheses': 1,  # zero-start root; raise for an explicit coarse root-seed experiment
    'proximal_iterations': 48,  # early proximal-bone steps; motion root is frozen here
    'articulated_iterations': 72,  # full-bone geometry steps
    'fine_iterations': 100,  # final small-step geometry refinement
    'registration_learning_rate': 0.025,  # static registration optimizer rate
    'articulated_learning_rate': 0.012,  # proximal/full articulation rate
    'fine_learning_rate': 0.004,  # fine-stage rate
    'registration_rotation_limit_deg': 25.0,  # residual static correction around the supplied profile camera
    'motion_rotation_limit_deg': 180.0,  # per-frame action-root range
    'joint_limit_deg': 180.0,  # global Meshy joint envelope
    'registration_translation_body_lengths': 0.25,  # residual static framing radius; camera fit owns the main placement
    'motion_translation_body_lengths': 0.65,  # per-frame body translation radius
    'registration_scale_log_limit': 0.25,  # residual static scale correction around the camera fit
    'proximal_depth': 2,  # hierarchy depth unlocked in the proximal stage
    'raster_antialias': True,  # exact CUDA coverage antialiasing
    'silhouette_weight': 0.0,  # exact alpha IoU
    'missing_weight': 0.35,  # target pixels absent in candidate
    'excess_weight': 0.35,  # candidate pixels outside target
    'edge_weight': 0.20,  # thin-limb edge agreement
    'sdf_weight': 0.20,  # long-range boundary guidance
    'depth_weight': 0.15,  # used when target depth is supplied
    'centroid_weight': 0.08,  # normalized centroid residual
    'occupancy_weight': 0.08,  # foreground-area residual
    'anatomy_weight': 0.08,  # discovered feature-point correspondence
    'rigidity_weight': 0.25,  # rigid prop/armor stability
    'pose_prior_weight': 0.015,  # unsupported-bone damping
    'velocity_weight': 0.025,  # robust frame-to-frame motion
    'acceleration_weight': 0.018,  # robust frame-to-frame acceleration
    'root_prior_weight': 0.025,  # motion-root damping
    'loop_weight': 0.002,  # optional loop seam regularization
    'loop_objective_enabled': False,  # enable only for cyclic source animations
    'robust_scale_deg': 48.0,  # Huber-like angular saturation scale
    'semantic_enabled': True,  # global SigLIP finalist scoring
    'semantic_model_id': 'google/siglip2-base-patch16-512',  # global encoder
    'semantic_layers': ('projected', 'pooler'),  # averaged global layers
    'dense_semantic_enabled': True,  # dense finalist scoring
    'dense_local_radius': 1,  # local patch search radius
    'dense_position_weight': 0.15,  # positional correspondence cost
    'front_view_enabled': True,  # use generated front views as auxiliary joint-disambiguation evidence
    'front_view_silhouette_weight': 0.00,  # canonicalized front alpha overlap; profile target remains authoritative
    'front_view_missing_weight': 0.08,  # penalize missing front-view subject regions
    'front_view_excess_weight': 0.08,  # penalize extra front-view regions
    'front_view_edge_weight': 0.06,  # canonicalized front boundary agreement
    'front_view_sdf_weight': 0.10,  # long-range canonical front boundary guidance
    'front_view_global_semantic_weight': 0.04,  # front-view global semantic arbitration weight
    'front_view_dense_semantic_weight': 0.08,  # foreground-masked front dense-feature weight
    'front_view_camera_scale': 1.0,  # initial orthographic front camera scale before the shared bind-pose fit
    'front_view_fit_camera_scale': True,  # fit one shared front camera scale from bind pose to the normalized reference extent
    'front_view_background_threshold': 12,  # RGB threshold used to separate generated subject from black background
    'front_view_subject_extent': 0.72,  # normalized maximum subject extent after generated-frame crop fitting
    'front_view_subject_padding': 0.08,  # context retained around each generated foreground bbox
    'semantic_rescue_iterations': 8,  # finalist rescue budget
    'global_semantic_weight': 0.08,  # low-weight gross visual rejection
    'dense_semantic_weight': 0.16,  # spatial finalist correspondence
    'semantic_microbatch': 1,  # finalist encoder batch; raise only after VRAM check
    # Match the proven single-joint notebook: global SigLIP2 plus gated DINOv3 dense tokens.
    # Change V2_DENSE_ENCODER to 'siglip2' for the same-model dense ablation.
    'dense_model_id': ('facebook/dinov3-vits16-pretrain-lvd1689m' if V2_DENSE_ENCODER == 'dinov3' else 'google/siglip2-base-patch16-512'),
    'dense_model_family': ('dino' if V2_DENSE_ENCODER == 'dinov3' else 'siglip2'),
    'dense_layers': ('last',),  # DINOv3 patch grid; hidden:<index> can be added for layer ablations
    'dense_local_files_only': True,  # use the already-downloaded checkpoint; set False only when allowing HF cache refreshes
    'interpolation_mode': 'quaternion_slerp',  # axis_angle_linear, quaternion_slerp, cubic_quaternion, or motion_aware
    'interpolation_output_fps': 60.0,  # final delivery cadence; source keyframes remain authentic DEF frames
    'interpolation_loop': False,  # add a final segment from the last keyframe back to frame 00
    'blender_lighting_mode': 'sun_above',  # final render lighting: H3-like sun or flat emission
    'blender_sun_energy': 2.0,  # directional sun intensity for the final Blender render
    'blender_sun_azimuth_deg': -35.0,  # sun direction around the world-up axis
    'blender_sun_elevation_deg': 55.0,  # sun height above the mesh
    'blender_sun_angle_deg': 5.0,  # soft shadow angle in degrees
    'blender_render_scale': 0.5,  # Blender parity/export resolution scale
    'target_depth_dir': None,  # prepared depth directory or None
    'checkpoint_every_stage': True,  # persist registration and stage checkpoints
    'progress_every': 1,  # live-dashboard update cadence
}
V2_CONFIG = preset_config(V2_PRESET, **V2_OVERRIDES)
V2_MODEL_PREFLIGHT = preflight(
    require_renderer=True,
    require_semantics=bool(V2_CONFIG.semantic_enabled or V2_CONFIG.dense_semantic_enabled),
    model_ids=tuple(dict.fromkeys(value for enabled, value in ((V2_CONFIG.semantic_enabled, V2_CONFIG.semantic_model_id), (V2_CONFIG.dense_semantic_enabled, V2_CONFIG.dense_model_id)) if enabled and value)),
    local_files_only_model_ids=((V2_CONFIG.dense_model_id,) if V2_CONFIG.dense_semantic_enabled and V2_CONFIG.dense_local_files_only else ()),
)
print('model preflight:', V2_MODEL_PREFLIGHT)
print(V2_CONFIG.to_jsonable())


In [ ]:
# Immutable preparation; no optimization occurs here.
# Re-read the selector so changing a dropdown does not require rerunning
# the widget cell. Reuse a rigidized mesh only if it belongs to this same
# currently selected source mesh.
_selected_v2 = _current_v2_selection()
_rigid_source = globals().get('_V2_RIGIDIZATION_SOURCE_MESH')
_rigid_mesh = globals().get('RIGIDIZED_MESH')
_rigid_report = globals().get('RIGIDIZATION_REPORT')
if (_rigid_source is not None and _rigid_mesh is not None and _rigid_report is not None
        and Path(_rigid_source).resolve() == Path(_selected_v2['mesh_path']).resolve()
        and Path(_rigid_mesh).is_file() and Path(_rigid_report).is_file()):
    _selected_v2['mesh_path'] = Path(_rigid_mesh).resolve()
    _selected_v2['rigidization_report'] = Path(_rigid_report).resolve()
V2_SELECTION = _selected_v2
SESSION = prepare_experiment(V2_SELECTION, V2_CONFIG)
print({'unit': V2_SELECTION.get('unit_name'), 'animation': V2_SELECTION.get('animation_name'), 'mesh': str(SESSION.rig.path), 'frames': list(SESSION.clip.frame_indices), 'source': str(SESSION.clip.source_dir), 'backend': 'nvdiffrast_cuda_triangle', 'rigFingerprint': SESSION.rig.fingerprint, 'clipFingerprint': SESSION.clip.fingerprint})


## Static registration pilot

Registration is mesh-to-camera alignment shared by the whole clip. It is not action motion and cannot be changed by later bone optimization.

In [ ]:
REGISTRATION_PREVIEW = registration_preview(SESSION)
for row in REGISTRATION_PREVIEW: print(row)


In [ ]:
# Single-line live dashboard; rerunning this cell starts a fresh solver state.
_progress_handle = display('starting V2...', display_id=True)
def _v2_progress(row):
    _vram = row.get('vram', {})
    _progress_handle.update(f"{row['stage']} {row['iteration']}/{row.get('iterations', '?')} | loss={row['bestLoss']:.5f} iou={row['bestIoU']:.4f} | accepted={row['acceptedHypotheses']}/{row['hypotheses']} rejected={row['rejectedHypotheses']} | bones={row.get('boneRotationMaxDeg', 0.0):.1f}deg root={row.get('motionRotationMaxDeg', 0.0):.1f}deg | worst=f{row.get('worstFrame')} step={row['stepScale']:.3f} rate={row.get('iterationRate', 0.0):.1f}/s VRAM={_vram.get('allocatedGiB', 0.0):.2f}G")
V2_RESULT = solve_clip(SESSION, progress_callback=_v2_progress)
print('selected:', V2_RESULT.provenance)


In [ ]:
# Exact CUDA preview, not the legacy soft point preview. Each frame is a
# 2x2 target/RGB/depth/IoU grid; the separate action view exposes hidden-axis twists.
V2_PREVIEW = preview(SESSION, V2_RESULT)
display(DisplayImage(filename=V2_PREVIEW['gif']))
display(DisplayImage(filename=V2_PREVIEW['actionCamera']))
if V2_PREVIEW.get('frontGif'):
    display(DisplayImage(filename=V2_PREVIEW['frontGif']))
print(V2_PREVIEW)


In [ ]:
# Canonical artifact consumed by the Blender parity/export step.
V2_RESULT_PATH = export(SESSION, V2_RESULT)
print(V2_RESULT_PATH)


## Blender parity gate and 60 fps delivery
This is export-only. It consumes the selected canonical V2 pose, renders the same registration and motion through Blender, compares keyframe alpha against the exact CUDA renderer, and writes GIF/MP4 only after all frames are present.

In [ ]:
# Export always writes the parity report. Keep enforcement explicit while the
# renderer/Blender raster thresholds are being calibrated on each unit.
V2_ENFORCE_PARITY = False  # set True only when the parity thresholds pass
V2_BLENDER_EXPORT = export_blender_60fps(SESSION, V2_RESULT)
print('parity:', V2_BLENDER_EXPORT['parity'])
print('gif:', V2_BLENDER_EXPORT['gif'])
print('mp4:', V2_BLENDER_EXPORT['mp4'])
V2_PARITY = require_blender_parity(V2_BLENDER_EXPORT) if V2_ENFORCE_PARITY else V2_BLENDER_EXPORT['parity']
display(DisplayImage(filename=V2_BLENDER_EXPORT['gif']))


In [ ]:
# Optional direct comparison with a frozen current-pipeline preview artifact.
# This is inspection-only and never feeds the V2 objective.
V2_BASELINE_PREVIEW_DIR = None  # set to a directory of baseline PNG frames, or leave None
if V2_BASELINE_PREVIEW_DIR is None:
    print('Set V2_BASELINE_PREVIEW_DIR to compare the frozen baseline against V2.')
else:
    _baseline_frames = sorted(Path(V2_BASELINE_PREVIEW_DIR).glob('*.png'))
    V2_COMPARISON = compare_preview_frames(_baseline_frames, V2_PREVIEW['frames'], Path(V2_RESULT_PATH).parent / 'baseline_vs_v2')
    print(V2_COMPARISON)
    display(DisplayImage(filename=V2_COMPARISON['gif']))
